In [1]:
import pandas as pd  # import biblioteki do tworzenia tabeli wyników

from tensorflow.keras.datasets import imdb  # import zbioru recenzji IMDB
from tensorflow.keras.preprocessing.sequence import pad_sequences  # import funkcji do wyrównywania długości sekwencji
from tensorflow.keras.models import Sequential  # import modelu sekwencyjnego
from tensorflow.keras.layers import Embedding, Flatten, Dense  # import warstw sieci neuronowej
from tensorflow.keras.regularizers import l2  # import regularyzacji L2

In [2]:
#funkcja budujaca model

In [3]:
def build_keras_model(input_dim, output_dim, sequence_length):  # funkcja tworząca model sieci neuronowej

    model = Sequential()  # utworzenie pustego modelu sekwencyjnego

    model.add(
        Embedding(
            input_dim=input_dim,  # liczba słów uwzględnianych przez model
            output_dim=output_dim,  # długość wektora reprezentującego każde słowo
            input_length=sequence_length  # długość każdej recenzji
        )
    )  # dodanie warstwy zamieniającej indeksy słów na wektory

    model.add(
        Flatten()
    )  # zamiana macierzy embeddingów na jednowymiarowy wektor

    model.add(
        Dense(
            1,  # jeden neuron wyjściowy dla klasyfikacji binarnej
            activation='sigmoid',  # zwrócenie prawdopodobieństwa klasy pozytywnej
            kernel_regularizer=l2(0.001)  # zastosowanie regularyzacji L2
        )
    )  # dodanie warstwy wyjściowej

    model.compile(
        optimizer='adam',  # wybór optymalizatora Adam
        loss='binary_crossentropy',  # funkcja straty dla klasyfikacji binarnej
        metrics=['accuracy']  # monitorowanie dokładności modelu
    )  # konfiguracja procesu uczenia

    return model  # zwrócenie przygotowanego modelu

In [4]:
# konfiguracja jiperparametrow ( pierwsza jest punktem odniesienia do tego z kursu)


In [5]:
configurations = [  # lista konfiguracji hiperparametrów

    {
        'num_words': 5000,  # liczba najczęściej występujących słów
        'maxlen': 200,  # maksymalna długość recenzji
        'embedding_dim': 16  # długość wektora reprezentującego słowo
    },

    {
        'num_words': 10000,  # zwiększenie liczby słów w słowniku
        'maxlen': 200,  # pozostawienie długości recenzji bez zmian
        'embedding_dim': 16  # pozostawienie rozmiaru embeddingu bez zmian
    },

    {
        'num_words': 5000,  # pozostawienie liczby słów bez zmian
        'maxlen': 300,  # zwiększenie maksymalnej długości recenzji
        'embedding_dim': 16  # pozostawienie rozmiaru embeddingu bez zmian
    },

    {
        'num_words': 5000,  # pozostawienie liczby słów bez zmian
        'maxlen': 200,  # pozostawienie długości recenzji bez zmian
        'embedding_dim': 32  # zwiększenie wymiaru embeddingu
    },

    {
        'num_words': 10000,  # zwiększenie liczby słów w słowniku
        'maxlen': 300,  # zwiększenie długości recenzji
        'embedding_dim': 32  # zwiększenie wymiaru embeddingu
    }

]

In [6]:
# trenowanie modeli

In [7]:
results = []  # utworzenie pustej listy do przechowywania wyników

for config in configurations:  # przejście przez wszystkie konfiguracje hiperparametrów

    num_words = config['num_words']  # pobranie liczby słów dla bieżącej konfiguracji
    maxlen = config['maxlen']  # pobranie maksymalnej długości recenzji
    embedding_dim = config['embedding_dim']  # pobranie wymiaru embeddingu

    print('\n' + '=' * 60)  # wyświetlenie separatora pomiędzy modelami

    print(
        f'num_words={num_words}, maxlen={maxlen}, embedding_dim={embedding_dim}'
    )  # wyświetlenie aktualnej konfiguracji

    (x_train, y_train), (x_test, y_test) = imdb.load_data(
        num_words=num_words
    )  # ponowne wczytanie danych z odpowiednim limitem słów

    x_train = pad_sequences(
        x_train,
        maxlen=maxlen
    )  # dopasowanie długości recenzji treningowych

    x_test = pad_sequences(
        x_test,
        maxlen=maxlen
    )  # dopasowanie długości recenzji testowych

    model = build_keras_model(
        input_dim=num_words,  # liczba słów przekazywana do warstwy Embedding
        output_dim=embedding_dim,  # wymiar wektora każdego słowa
        sequence_length=maxlen  # długość recenzji
    )  # utworzenie nowego modelu dla bieżącej konfiguracji

    history = model.fit(
        x_train,  # dane treningowe
        y_train,  # etykiety treningowe
        epochs=5,  # liczba przejść przez cały zbiór treningowy
        batch_size=32,  # liczba próbek przetwarzanych jednocześnie
        validation_split=0.2,  # przeznaczenie 20% danych treningowych do walidacji
        verbose=1  # wyświetlanie przebiegu uczenia
    )  # wytrenowanie modelu

    test_loss, test_accuracy = model.evaluate(
        x_test,  # dane testowe
        y_test,  # prawdziwe etykiety testowe
        verbose=0  # wyłączenie dodatkowego paska postępu
    )  # ocena modelu na zbiorze testowym

    train_accuracy = history.history['accuracy'][-1]  # dokładność treningowa z ostatniej epoki
    validation_accuracy = history.history['val_accuracy'][-1]  # dokładność walidacyjna z ostatniej epoki

    results.append({
        'num_words': num_words,  # zapisanie liczby słów
        'maxlen': maxlen,  # zapisanie maksymalnej długości recenzji
        'embedding_dim': embedding_dim,  # zapisanie wymiaru embeddingu
        'train_accuracy': train_accuracy,  # zapisanie dokładności treningowej
        'validation_accuracy': validation_accuracy,  # zapisanie dokładności walidacyjnej
        'test_accuracy': test_accuracy,  # zapisanie dokładności testowej
        'test_loss': test_loss  # zapisanie wartości funkcji straty
    })  # dodanie wyników bieżącego modelu do listy

    print(
        f'Test accuracy: {test_accuracy:.4f}'
    )  # wyświetlenie dokładności bieżącego modelu


num_words=5000, maxlen=200, embedding_dim=16


C:\Users\micha\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\embedding.py:123: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 10ms/step - accuracy: 0.7387 - loss: 0.5308 - val_accuracy: 0.8430 - val_loss: 0.3791
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.8824 - loss: 0.3141 - val_accuracy: 0.8700 - val_loss: 0.3402
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 11s 10ms/step - accuracy: 0.9126 - loss: 0.2607 - val_accuracy: 0.8740 - val_loss: 0.3318
Epoch 4/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 10s 9ms/step - accuracy: 0.9355 - loss: 0.2259 - val_accuracy: 0.8714 - val_loss: 0.3384
Epoch 5/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 6s 9ms/step - accuracy: 0.9553 - loss: 0.1963 - val_accuracy: 0.8688 - val_loss: 0.3495
Test accuracy: 0.8678

num_words=10000, maxlen=200, embedding_dim=16
Epoch 1/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 9s 11ms/step - accuracy: 0.7327 - loss: 0.5357 - val_accuracy: 0.8600 - val_loss: 0.3610
Epoch 2/5
625/625 ━━━━━━━━━━━━━━━━━━━━ 7s 10ms/step - accuracy: 0.8951 - loss: 0.2932 - val_accuracy: 0.8696 - val_loss: 0.3345
Epoch 3/5
625/625 ━━━━━━━━━━━━━━━━━━

In [8]:
# wyniki

In [9]:
results_df = pd.DataFrame(results)  # zamiana listy wyników na tabelę DataFrame

results_df = results_df.sort_values(
    by='test_accuracy',
    ascending=False
)  # uporządkowanie modeli od najwyższej dokładności testowej

results_df  # wyświetlenie tabeli wyników

,num_words,maxlen,embedding_dim,train_accuracy,validation_accuracy,test_accuracy,test_loss
1,10000,200,16,0.97600,0.8764,0.87300,0.347376
2,5000,300,16,0.94805,0.8720,0.87156,0.342777
4,10000,300,32,0.99135,0.8780,0.87068,0.355118
0,5000,200,16,0.95530,0.8688,0.86784,0.352103
3,5000,200,32,0.98025,0.8578,0.85220,0.393898


Jak analizować wyniki

num_words określa wielkość słownika. Większa wartość pozwala modelowi uwzględnić więcej rzadziej występujących słów, ale zwiększa liczbę parametrów warstwy Embedding.

maxlen określa liczbę słów zachowanych z każdej recenzji. Większa wartość dostarcza modelowi więcej tekstu, ale zwiększa liczbę danych przekazywanych do warstwy Dense.

embedding_dim określa długość wektora opisującego każde słowo. Większa wartość pozwala utworzyć bogatszą reprezentację słów, ale zwiększa złożoność modelu i ryzyko przeuczenia.

# wniosek

Przeprowadzono eksperyment polegający na zmianie wartości hiperparametrów `num_words`, `maxlen` oraz `embedding_dim` i porównano ich wpływ na skuteczność klasyfikacji recenzji. Najlepsze wyniki uzyskał model z parametrami **num_words = 10000**, **maxlen = 200** oraz **embedding_dim = 16**, osiągając dokładność **87,30%** na zbiorze testowym.

Zwiększenie liczby słów (`num_words`) z 5000 do 10000 poprawiło skuteczność modelu, ponieważ sieć mogła wykorzystać bogatszy słownik. Zwiększenie długości analizowanej recenzji (`maxlen`) z 200 do 300 słów również przyniosło niewielką poprawę wyników, umożliwiając modelowi analizę większej ilości informacji zawartych w tekście.

Natomiast zwiększenie wymiaru embeddingu (`embedding_dim`) z 16 do 32 nie poprawiło jakości klasyfikacji. Modele z większym embeddingiem osiągały bardzo wysoką skuteczność na zbiorze treningowym (do 99,14%), jednak niższą na zbiorze testowym, co świadczy o przeuczeniu (overfittingu). Oznacza to, że bardziej złożony model lepiej zapamiętywał dane treningowe, ale gorzej uogólniał wiedzę na nowe przykłady.

Uzyskane wyniki pokazują, że zwiększanie liczby parametrów modelu nie zawsze prowadzi do poprawy skuteczności. Kluczowe jest dobranie hiperparametrów zapewniających równowagę pomiędzy zdolnością uczenia a możliwością poprawnego klasyfikowania wcześniej niewidzianych danych.
